## Classification Accuracy

In [1]:
# from typing import Iterable, Mapping, Sequence
# import numpy as np
# from sklearn.preprocessing import MultiLabelBinarizer
# from sklearn.metrics import (
#     jaccard_score,
#     precision_score,
#     recall_score,
#     f1_score,
#     hamming_loss,
#     accuracy_score,
# )

# def _normalize_label_set(x, sep=";"):
#     """
#     Normalize one entry into a Python set of strings.
#     """
#     if x is None:
#         return set()

#     if isinstance(x, (set, list, tuple)):
#         return {str(v).strip() for v in x if str(v).strip()}

#     s = str(x).strip()
#     if not s:
#         return set()

#     return {t.strip() for t in s.split(sep) if t.strip()}


# def _align_inputs(y_true, y_pred):
#     """
#     Align inputs:
#     - dicts: align on key intersection
#     - lists/sequences: align by order
#     """
#     if isinstance(y_true, Mapping) and isinstance(y_pred, Mapping):
#         common_keys = y_true.keys() & y_pred.keys()
#         y_true = [y_true[k] for k in common_keys]
#         y_pred = [y_pred[k] for k in common_keys]
#     else:
#         if len(y_true) != len(y_pred):
#             raise ValueError("List inputs must have the same length")
#     return y_true, y_pred


# def _binarize(y_true, y_pred, sep=";"):
#     """
#     Convert aligned multilabel data into binary matrices.
#     """
#     y_true, y_pred = _align_inputs(y_true, y_pred)

#     y_true_sets = [_normalize_label_set(x, sep) for x in y_true]
#     y_pred_sets = [_normalize_label_set(x, sep) for x in y_pred]

#     labels = sorted(set().union(*y_true_sets).union(*y_pred_sets))
#     mlb = MultiLabelBinarizer(classes=labels)

#     Y_true = mlb.fit_transform(y_true_sets)
#     Y_pred = mlb.transform(y_pred_sets)

#     return Y_true, Y_pred


# def jaccard_samples(y_true, y_pred, sep=";"):
#     """
#     Mean Jaccard similarity across samples.
#     """
#     Y_true, Y_pred = _binarize(y_true, y_pred, sep)
#     return jaccard_score(Y_true, Y_pred, average="samples", zero_division=0)

# def multilabel_prf(y_true, y_pred, average="micro", sep=";"):
#     """
#     Precision, Recall, F1 for multilabel data.
#     average ∈ {'micro', 'macro', 'samples'}
#     """
#     Y_true, Y_pred = _binarize(y_true, y_pred, sep)

#     return {
#         "precision": precision_score(Y_true, Y_pred, average=average, zero_division=0),
#         "recall": recall_score(Y_true, Y_pred, average=average, zero_division=0),
#         "f1": f1_score(Y_true, Y_pred, average=average, zero_division=0),
#     }

# def multilabel_hamming_loss(y_true, y_pred, sep=";"):
#     """
#     Fraction of incorrect label assignments.
#     """
#     Y_true, Y_pred = _binarize(y_true, y_pred, sep)
#     return hamming_loss(Y_true, Y_pred)


# def subset_accuracy(y_true, y_pred, sep=";"):
#     """
#     Fraction of samples whose predicted label set
#     exactly matches the true label set.
#     """
#     Y_true, Y_pred = _binarize(y_true, y_pred, sep)
#     return accuracy_score(Y_true, Y_pred)


In [2]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "eval").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from eval.classification.metrics import (
    jaccard_samples,
    multilabel_hamming_loss,
    multilabel_prf,
    subset_accuracy,
)



In [3]:
# import pandas as pd
# df_gt = pd.read_csv(
#     "sampled_papers_full.csv",
#     index_col=False, 
#     sep="\t"
#     )


In [12]:
from pathlib import Path

import pandas as pd

from eval.classification.error_analysis import per_run_metrics
from eval.classification.summary import summarize_per_run_metrics

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "eval").exists() else NOTEBOOK_DIR.parent
GT_PATH = REPO_ROOT / "sampled_papers_full.csv"
DEFAULT_RUN_ROOT = REPO_ROOT / "outputs" / "nemotron_dense"

TYPE_PREFIX_TO_TASK = {
    "AVAILABILITY": "data-accessibility",
    "GEO": "geo",
    "DATA_TYPE": "data-type",
    "PTYPE": "paper-type",
}


def evaluate_classification(
    type_prefix,
    model_name="gemini-2-5-flash",
    temperature="1.0",
    run_root=None,
    show_variance=True,
):
    assert type_prefix in TYPE_PREFIX_TO_TASK, "Invalid type prefix"

    task = TYPE_PREFIX_TO_TASK[type_prefix]
    run_root = Path(run_root) if run_root is not None else DEFAULT_RUN_ROOT

    print(
        f"\n-----------------------------------------------------"
        f"\n{type_prefix} results for {model_name} with temperature = {temperature}"
        f"\nrun_root = {run_root}\n"
    )

    per_run = per_run_metrics(
        ground_truth_path=GT_PATH,
        run_roots=[run_root],
    )
    if per_run.empty:
        print("[WARN] No result files were discovered.")
        return None

    filtered = per_run[
        (per_run["task"] == task)
        & (per_run["model"] == model_name)
        & (per_run["temperature"].astype(str) == str(temperature))
    ].copy()

    if filtered.empty:
        print("[WARN] No matching result files found for this configuration.")
        return None

    summary = summarize_per_run_metrics(filtered)
    row = summary.iloc[0]

    def _fmt(metric_name: str) -> str:
        return (
            f"mean={row[f'{metric_name}_mean']:.4f} "
            f"+-{row[f'{metric_name}_std']:.4f} "
            f"[{row[f'{metric_name}_min']:.4f}, {row[f'{metric_name}_max']:.4f}]"
        )

    print(f"Files evaluated: {int(row['n_files'])}")
    print(f"Jaccard(samples): {_fmt('jaccard_samples')}")
    print(
        "PRF micro: "
        f"precision={row['micro_precision_mean']:.4f}+-{row['micro_precision_std']:.4f} "
        f"recall={row['micro_recall_mean']:.4f}+-{row['micro_recall_std']:.4f} "
        f"f1={row['micro_f1_mean']:.4f}+-{row['micro_f1_std']:.4f}"
    )
    print(
        "PRF macro: "
        f"precision={row['macro_precision_mean']:.4f}+-{row['macro_precision_std']:.4f} "
        f"recall={row['macro_recall_mean']:.4f}+-{row['macro_recall_std']:.4f} "
        f"f1={row['macro_f1_mean']:.4f}+-{row['macro_f1_std']:.4f}"
    )
    print(f"Subset accuracy: {_fmt('subset_accuracy')}")
    if show_variance:
        print(f"Hamming loss: {_fmt('hamming_loss')}")

    display_cols = [
        "source_file",
        "jaccard_samples",
        "micro_f1",
        "macro_f1",
        "subset_accuracy",
        "hamming_loss",
        "n_predictions",
    ]
    display(filtered[display_cols].reset_index(drop=True))
    return filtered


show_variance = False
for model_name in [
    # "gemini-2-5-pro",
    # "gemini-2-5-flash",

    "nvidia-nemotron-3-super-120b-a12b-free"
]:
    for temperature in [
        # "1.0",
        "0.0",
    ]:
        for task in TYPE_PREFIX_TO_TASK.keys(): 
            evaluate_classification(
                task,
                model_name=model_name,
                temperature=temperature,
                run_root=DEFAULT_RUN_ROOT,
                show_variance=show_variance,
            )




-----------------------------------------------------
AVAILABILITY results for nvidia-nemotron-3-super-120b-a12b-free with temperature = 0.0
run_root = /Users/vins/Documents/Projects/EpiScope/outputs/nemotron_dense

Files evaluated: 1
Jaccard(samples): mean=0.5292 +-0.0000 [0.5292, 0.5292]
PRF micro: precision=0.5372+-0.0000 recall=0.4594+-0.0000 f1=0.4952+-0.0000
PRF macro: precision=0.2991+-0.0000 recall=0.2333+-0.0000 f1=0.2386+-0.0000
Subset accuracy: mean=0.5187 +-0.0000 [0.5187, 0.5187]


,source_file,jaccard_samples,micro_f1,macro_f1,subset_accuracy,hamming_loss,n_predictions
0,/Users/vins/Documents/Projects/EpiScope/output...,0.529206,0.495238,0.238605,0.518692,0.05384,214



-----------------------------------------------------
GEO results for nvidia-nemotron-3-super-120b-a12b-free with temperature = 0.0
run_root = /Users/vins/Documents/Projects/EpiScope/outputs/nemotron_dense

Files evaluated: 1
Jaccard(samples): mean=0.8629 +-0.0000 [0.8629, 0.8629]
PRF micro: precision=0.7695+-0.0000 recall=0.8798+-0.0000 f1=0.8210+-0.0000
PRF macro: precision=0.5160+-0.0000 recall=0.6767+-0.0000 f1=0.5570+-0.0000
Subset accuracy: mean=0.8598 +-0.0000 [0.8598, 0.8598]


,source_file,jaccard_samples,micro_f1,macro_f1,subset_accuracy,hamming_loss,n_predictions
0,/Users/vins/Documents/Projects/EpiScope/output...,0.862928,0.820976,0.55695,0.859813,0.018505,214



-----------------------------------------------------
DATA_TYPE results for nvidia-nemotron-3-super-120b-a12b-free with temperature = 0.0
run_root = /Users/vins/Documents/Projects/EpiScope/outputs/nemotron_dense

Files evaluated: 1
Jaccard(samples): mean=0.7560 +-0.0000 [0.7560, 0.7560]
PRF micro: precision=0.7436+-0.0000 recall=0.6797+-0.0000 f1=0.7102+-0.0000
PRF macro: precision=0.3718+-0.0000 recall=0.2097+-0.0000 f1=0.2306+-0.0000
Subset accuracy: mean=0.7523 +-0.0000 [0.7523, 0.7523]


,source_file,jaccard_samples,micro_f1,macro_f1,subset_accuracy,hamming_loss,n_predictions
0,/Users/vins/Documents/Projects/EpiScope/output...,0.755997,0.710204,0.230624,0.752336,0.047397,214



-----------------------------------------------------
PTYPE results for nvidia-nemotron-3-super-120b-a12b-free with temperature = 0.0
run_root = /Users/vins/Documents/Projects/EpiScope/outputs/nemotron_dense

Files evaluated: 1
Jaccard(samples): mean=0.8505 +-0.0000 [0.8505, 0.8505]
PRF micro: precision=0.8505+-0.0000 recall=0.8505+-0.0000 f1=0.8505+-0.0000
PRF macro: precision=0.8179+-0.0000 recall=0.6612+-0.0000 f1=0.7091+-0.0000
Subset accuracy: mean=0.8505 +-0.0000 [0.8505, 0.8505]


,source_file,jaccard_samples,micro_f1,macro_f1,subset_accuracy,hamming_loss,n_predictions
0,/Users/vins/Documents/Projects/EpiScope/output...,0.850467,0.850467,0.709089,0.850467,0.049844,214


### Image

In [5]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
from collections import Counter



def _read_results_with_fallback(path: str, seps=(";", "\t")) -> tuple[pd.DataFrame, str]:
    """
    Try reading results file with separators in order.
    Success is defined as: dataframe contains 'paper_id' and 'classification' columns.
    Returns (df, used_sep). Raises ValueError if all attempts fail.
    """
    last_err = None
    for sep in seps:
        try:
            df = pd.read_csv(path, index_col=False, sep=sep)
            if {"paper_id", "classification"}.issubset(df.columns):
                return df, sep
            else:
                last_err = ValueError(
                    f"Loaded but missing required columns with sep={repr(sep)}. "
                    f"Columns: {list(df.columns)}"
                )
        except Exception as e:
            last_err = e

    raise ValueError(f"Could not parse '{path}' with seps {list(map(repr, seps))}. Last error: {last_err}")


df_gt = pd.read_csv("../sampled_papers_full.csv", index_col=False, sep="\t")

map_to_foldname = {
    "AVAILABILITY": "data-accessibility",
    "GEO": "geo",
    "DATA_TYPE": "data-type",
    "PTYPE": "paper-type",
}

model_name = "gemini-2-5-flash"
temperature = "0.0"

for type_prefix in [
    "PTYPE",
    # "AVAILABILITY", 
    "GEO", 
    "DATA_TYPE", 
    ]:

    base_fold_str = "output"
    search_path = f"{base_fold_str}/{map_to_foldname[type_prefix]}/{model_name}/temperature_{temperature}/**/final_*.tsv"
    result_files = glob.glob(search_path, recursive=True)

    if not result_files:
        print(f"[WARN] No result files found for {type_prefix} at: {search_path}")

    # compute GT dict once
    gt_col = f"{type_prefix.lower()}_classification"
    if gt_col not in df_gt.columns:
        raise KeyError(f"GT file missing expected column '{gt_col}'. Available: {list(df_gt.columns)}")

    gt_dict_str = df_gt.set_index("paper_id")[gt_col].astype(str).to_dict()

    for file in result_files:
        try:
            df_class, used_sep = _read_results_with_fallback(file, seps=("\t",))
        except Exception as e:
            print(f"[FAIL] {file}: tried '\\t' but couldn't parse. ({type(e).__name__}: {e})")
            continue


    entries = [data_type for data_types_list in df_class.classification.values for data_type in eval(data_types_list)]
    c = Counter(entries)

    n_papers = len(df_class)
    c_frac = {k:v/n_papers for k,v in c.items()}

    data_series = pd.Series(c).sort_values(ascending=False)

    class_to_label = {x : x for x in c_frac.keys()}



    plt.figure(figsize=(4, 3))
    plt.title(type_prefix, fontsize=12, pad=5)
    bars = plt.barh(
        [class_to_label[i] for i in data_series.index], 
        data_series.values,
        edgecolor='black', 
        alpha=0.7
    )


    # Apply specific styles to "Unclear", "No Data", and "Synthetic"
    for bar, label in zip(bars, data_series.index):
        if label in []: #['UNCLEAR']:
            bar.set_color('gray')  # Gray out the bars
            bar.set_hatch('//')    # Add a hatch pattern for better distinction

    # Add value labels to the end of each bar for clarity
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 10,  # Position x-coordinate
                bar.get_y() + bar.get_height() / 2,  # Position y-coordinate
                f'{width:}',  # Format as percentage
                ha='left',  # Horizontal alignment
                va='center',  # Vertical alignment
                fontsize=10)

    # Remove the top and right spines
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)

    # Ensure the layout is clean
    plt.tight_layout()
    plt.show()


[WARN] No result files found for PTYPE at: output/paper-type/gemini-2-5-flash/temperature_0.0/**/final_*.tsv


NameError: name 'df_class' is not defined

In [6]:
import ast
import glob
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd


def read_results(path: str) -> pd.DataFrame:
    """Read a tab-separated results file and validate required columns."""
    df = pd.read_csv(path, sep="\t", index_col=False)

    required = {"paper_id", "classification"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing required columns in '{path}': {sorted(missing)}. "
            f"Found: {list(df.columns)}"
        )

    return df


def count_classifications(df: pd.DataFrame) -> pd.Series:
    """Flatten classification lists and count label frequencies."""
    entries = [
        label
        for values in df["classification"].dropna()
        for label in ast.literal_eval(values)
    ]
    return pd.Series(Counter(entries)).sort_values(ascending=False)


df_gt = pd.read_csv("sampled_papers_full.csv", sep="\t", index_col=False)

map_to_foldname = {
    "AVAILABILITY": "data-accessibility",
    "GEO": "geo",
    "DATA_TYPE": "data-type",
    "PTYPE": "paper-type",
}

model_name = "gemini-2-5-flash"
temperature = "0.0"

type_prefixes = [
    "PTYPE",
    # "AVAILABILITY",
    "GEO",
    "DATA_TYPE",
]

results_by_type = {}

for type_prefix in type_prefixes:
    search_path = (
        f"output_full/{map_to_foldname[type_prefix]}/{model_name}/"
        f"temperature_{temperature}/**/final_*.tsv"
    )
    result_files = glob.glob(search_path, recursive=True)

    if not result_files:
        print(f"[WARN] No result files found for {type_prefix} at: {search_path}")
        continue

    gt_col = f"{type_prefix.lower()}_classification"
    if gt_col not in df_gt.columns:
        raise KeyError(
            f"GT file missing expected column '{gt_col}'. "
            f"Available: {list(df_gt.columns)}"
        )

    frames = []
    for file in result_files:
        try:
            df = read_results(file)
            print(f"[OK] Successfully read '{file}' with {len(df)} entries.")
            frames.append(read_results(file))
        except Exception as e:
            print(f"[FAIL] {file}: {type(e).__name__}: {e}")

    if not frames:
        print(f"[WARN] No valid parsed files for {type_prefix}")
        continue

    df_class = pd.concat(frames, ignore_index=True)
    results_by_type[type_prefix] = count_classifications(df_class)


if results_by_type:
    fig, axes = plt.subplots(1, len(results_by_type), figsize=(5 * len(results_by_type), 4))

    if len(results_by_type) == 1:
        axes = [axes]

    for ax, (type_prefix, data_series) in zip(axes, results_by_type.items()):
        bars = ax.barh(
            data_series.index,
            data_series.values,
            edgecolor="black",
            alpha=0.7,
        )

        for bar, label in zip(bars, data_series.index):
            if label in []:  # e.g. ["UNCLEAR"]
                bar.set_color("gray")
                bar.set_hatch("//")

        for bar in bars:
            width = bar.get_width()
            ax.text(
                width + max(data_series.values) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{int(width)}",
                ha="left",
                va="center",
                fontsize=9,
            )

        ax.set_title(type_prefix, fontsize=12, pad=5)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.invert_yaxis()  # highest count on top

    fig.tight_layout()
    plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'sampled_papers_full.csv'

In [7]:
import ast
import glob
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd


def read_results(path: str) -> pd.DataFrame:
    """Read a tab-separated results file and validate required columns."""
    df = pd.read_csv(path, sep="\t", index_col=False)

    required = {"paper_id", "classification"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing required columns in '{path}': {sorted(missing)}. "
            f"Found: {list(df.columns)}"
        )

    return df


def summarize_classifications(df: pd.DataFrame) -> pd.DataFrame:
    """Return count and fraction per label."""
    entries = [
        label
        for values in df["classification"].dropna()
        for label in ast.literal_eval(values)
    ]

    counts = pd.Series(Counter(entries), name="count").sort_values(ascending=False)
    out = counts.to_frame()
    out["fraction"] = out["count"] / len(df)
    out["pct"] = out["fraction"] * 100
    return out


def shorten_label(label: str) -> str:
    label = label.replace("NON_TRADITIONAL", "NTD")
    label = label.replace("_", " ")
    label = label.lower()
    #capitalize first letter of each word
    label = label.title()
    return label.replace("Empirical ","")




df_gt = pd.read_csv("sampled_papers_full.csv", sep="\t", index_col=False)

map_to_foldname = {
    "AVAILABILITY": "data-accessibility",
    "GEO": "geo",
    "DATA_TYPE": "data-type",
    "PTYPE": "paper-type",
}

model_name = "gemini-2-5-flash"
temperature = "0.0"

type_prefixes = [
    "PTYPE",
    # "AVAILABILITY",
    "GEO",
    "DATA_TYPE",
]

results_by_type = {}

for type_prefix in type_prefixes:
    search_path = (
        f"output_full/{map_to_foldname[type_prefix]}/{model_name}/"
        f"temperature_{temperature}/**/final_*.tsv"
    )
    result_files = glob.glob(search_path, recursive=True)

    if not result_files:
        print(f"[WARN] No result files found for {type_prefix} at: {search_path}")
        continue

    gt_col = f"{type_prefix.lower()}_classification"
    if gt_col not in df_gt.columns:
        raise KeyError(
            f"GT file missing expected column '{gt_col}'. "
            f"Available: {list(df_gt.columns)}"
        )

    frames = []
    for file in result_files:
        try:
            frames.append(read_results(file))
        except Exception as e:
            print(f"[FAIL] {file}: {type(e).__name__}: {e}")

    if not frames:
        print(f"[WARN] No valid parsed files for {type_prefix}")
        continue

    df_class = pd.concat(frames, ignore_index=True)

    results_by_type[type_prefix] = {
        "summary": summarize_classifications(df_class),
        "n_files": len(frames),
        "n_papers": len(df_class),
    }

titles_map = {
    "PTYPE": "Paper Type",
    "AVAILABILITY": "Data Accessibility",
    "GEO": "Geographical Scope",
    "DATA_TYPE": "Data Type",
}

if results_by_type:
    fig, axes = plt.subplots(
        1,
        len(results_by_type),
        figsize=(3.5 * len(results_by_type), 3),
        sharex=False,
        constrained_layout=True,
    )

    if len(results_by_type) == 1:
        axes = [axes]

    for ax, (type_prefix, info) in zip(axes, results_by_type.items()):
        summary = info["summary"].copy()
        summary["label"] = [shorten_label(label) for label in summary.index]

        n_files = info["n_files"]
        n_papers = info["n_papers"]

        bars = ax.barh(
            range(len(summary))[::-1],
            summary["pct"].values[::-1],
            edgecolor="black",
            alpha=0.8,
        )

        ax.set_yticks(range(len(summary))[::-1])
        ax.set_yticklabels(summary["label"].values[::-1], fontsize=10)

        for bar, (_, row) in zip(bars, summary.iloc[::-1].iterrows()):
            x = bar.get_width()
            y = bar.get_y() + bar.get_height() / 2

            ax.text(
                x + 1.0,
                y + 0.15,
                f"{row['pct']:.1f}%",
                va="center",
                fontsize=9.5,
            )

            ax.text(
                x + 1.0,
                y - 0.25,
                f"n={int(row['count'])}",
                va="center",
                fontsize=7.5,
                color="gray",
            )

        ax.set_title(
            f"{titles_map.get(type_prefix, type_prefix)}",
            fontsize=11,
            pad=8,
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(axis="x", linestyle=":", alpha=0.4)

        xmax = max(summary["pct"].max() * 1.22, 10)
        ax.set_xlim(0, xmax)
plt.tight_layout()
# plt.show()
plt.savefig("classification_full_summary.pdf", dpi=300)

FileNotFoundError: [Errno 2] No such file or directory: 'sampled_papers_full.csv'

## Stability and error analysis

In [8]:
"""
Structured Error Analysis
==========================
  A. Within-config consistency
  B. Containment
  C. Pro @ T=0 always wrong
"""

from pathlib import Path

from eval.classification.error_analysis import (
    containment_pro_vs_flash,
    load_merged_predictions,
    pro_always_wrong,
    within_config_consistency,
)

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "eval").exists() else NOTEBOOK_DIR.parent
GT_PATH = REPO_ROOT / "sampled_papers_full.csv"
OUTPUT_DIR = REPO_ROOT / "outputs" / "output"
MODEL_PRO = "gemini-2-5-pro"
MODEL_FLASH = "gemini-2-5-flash"
FIXED_TEMP = "0.0"

merged = load_merged_predictions(
    ground_truth_path=GT_PATH,
    run_roots=[OUTPUT_DIR],
)

print("\n=== A. Within-config consistency across runs ===\n")
consistency = within_config_consistency(merged)
display(consistency)
consistency.to_csv(REPO_ROOT / "within_config_consistency.csv", index=False)

print(f"\n=== B. Containment: pro vs flash (temperature = {FIXED_TEMP}) ===\n")
containment = containment_pro_vs_flash(
    merged,
    temperature=FIXED_TEMP,
    pro_model=MODEL_PRO,
    flash_model=MODEL_FLASH,
)
display(containment)
containment.to_csv(REPO_ROOT / "containment_pro_vs_flash.csv", index=False)

print(f"\n=== C. Papers pro (T={FIXED_TEMP}) gets wrong in every run ===\n")
pro_errors = pro_always_wrong(
    merged,
    temperature=FIXED_TEMP,
    pro_model=MODEL_PRO,
)
pro_errors.to_csv(REPO_ROOT / "pro_always_wrong.csv", index=False)

for task, grp in pro_errors.groupby("task"):
    print(f"\n-- {task} ({len(grp)} papers always wrong) --")
    display(
        grp.drop(columns="task")
           .assign(
               gt_label=lambda d: d["gt_set"].map(lambda s: list(sorted(s))),
               top_pred=lambda d: d["top_wrong_pred"].map(lambda s: list(sorted(s))),
           )
           .drop(columns=["gt_set", "top_wrong_pred"])
           .reset_index(drop=True)
    )




=== A. Within-config consistency across runs ===



,task,model,temperature,n_papers,n_stable,n_unstable,n_stable_wrong,n_stable_correct,pct_stable
0,data-accessibility,gemini-2-5-flash,0.0,214,143,71,52,91,66.8
1,data-accessibility,gemini-2-5-flash,1.0,214,69,145,18,51,32.2
2,data-accessibility,gemini-2-5-pro,0.0,214,136,78,17,119,63.6
3,data-accessibility,gemini-2-5-pro,1.0,214,110,104,12,98,51.4
4,data-type,gemini-2-5-flash,0.0,214,203,11,21,182,94.9
5,data-type,gemini-2-5-flash,1.0,214,191,23,17,174,89.3
6,data-type,gemini-2-5-pro,0.0,214,191,23,2,189,89.3
7,data-type,gemini-2-5-pro,1.0,214,193,21,9,184,90.2
8,paper-type,gemini-2-5-flash,0.0,214,192,22,15,177,89.7
9,paper-type,gemini-2-5-flash,1.0,214,163,51,3,160,76.2



=== B. Containment: pro vs flash (temperature = 0.0) ===



,task,temperature,n_pro_errors,n_flash_errors,n_shared,pro_in_flash_pct,flash_in_pro_pct,jaccard_pct,n_pro_only,n_flash_only
0,data-accessibility,0.0,95,123,72,75.8,58.5,49.3,23,51
1,data-type,0.0,25,32,20,80.0,62.5,54.1,5,12
2,paper-type,0.0,34,37,15,44.1,40.5,26.8,19,22



=== C. Papers pro (T=0.0) gets wrong in every run ===


-- data-accessibility (33 papers always wrong) --


,paper_id,n_runs,wrong_runs,gt_label,top_pred
0,10.1007/s10661-016-5117-6,5,5,[OPEN],"[NOT_STATED, REFERENCED]"
1,10.1016/0378-1119(87)90166-1,5,5,"[REFERENCED, REPORTED]","[NOT_STATED, REFERENCED, REPORTED]"
2,10.1016/j.ijid.2013.07.015,5,5,"[OPEN, REFERENCED, REPORTED]","[REFERENCED, REPORTED]"
3,10.1016/j.ijid.2020.04.055,5,5,[REPORTED],[REFERENCED]
4,10.1016/s1386-6532(02)00268-8,5,5,[REPORTED],"[REFERENCED, REPORTED]"
5,10.1016/s1473-3099(20)30314-5,5,5,"[OPEN, REPORTED]",[REPORTED]
6,10.1038/nature04795,5,5,"[AVAILABLE_UPON_REQUEST, REFERENCED, REPORTED]","[AVAILABLE_UPON_REQUEST, REFERENCED]"
7,10.1080/22221751.2020.1796528,5,5,[OPEN],"[NOT_STATED, REPORTED]"
8,10.1101/2020.01.23.20018549,5,5,"[NOT_STATED, OPEN, REFERENCED]",[REPORTED]
9,10.1101/2020.02.08.20021212,5,5,"[AVAILABLE_UPON_REQUEST, REPORTED]",[REPORTED]



-- data-type (3 papers always wrong) --


,paper_id,n_runs,wrong_runs,gt_label,top_pred
0,10.1093/infdis/jiw200,5,5,[NO_EMPIRICAL_DATA],[TRADITIONAL]
1,10.2471/blt.15.020415,5,5,[TRADITIONAL],"[NON_TRADITIONAL_HEALTH, TRADITIONAL]"
2,10.3934/mbe.2020173,5,5,[TRADITIONAL],"[SYNTHETIC, TRADITIONAL]"



-- paper-type (8 papers always wrong) --


,paper_id,n_runs,wrong_runs,gt_label,top_pred
0,10.1089/vbz.2013.1421,5,5,[EMPIRICAL],[INFERENCE]
1,10.1093/ofid/ofac101,5,5,[EMPIRICAL],[INFERENCE]
2,10.1101/2020.02.08.20021212,5,5,[EMPIRICAL],[INFERENCE]
3,10.11604/pamj.2024.47.22.42156,5,5,[EMPIRICAL],[INFERENCE]
4,10.15585/mmwr.mm6616a1,5,5,[EMPIRICAL],[INFERENCE]
5,10.3201/eid2111.150764,5,5,[EMPIRICAL],[INFERENCE]
6,10.4269/ajtmh.2010.09-0293,5,5,[EMPIRICAL],[INFERENCE]
7,10.7748/ns.29.14.16.s20,5,5,[EMPIRICAL],[INFERENCE]


In [10]:
"""
Structured Error Analysis
==========================
  A. Within-config consistency – does a model give the same answer across
                                  repeated runs on the same paper?

  B. Containment               – at a fixed temperature, how much of pro's
                                  errors are covered by flash's errors?

  C. Pro @ T=0 always wrong    – papers pro gets wrong in every run,
                                  per task (with the actual wrong prediction)
"""

from __future__ import annotations

import ast
import re
import unicodedata
from pathlib import Path

import pandas as pd


GT_PATH    = Path("../sampled_papers_full.csv")
OUTPUT_DIR = Path("output")

TASK_TO_GT_COL = {
    "geo":                "geo_classification",
    "data-type":          "data_type_classification",
    "data-accessibility": "availability_classification",
    "paper-type":         "ptype_classification",
}

MODEL_PRO   = "gemini-2-5-pro"
MODEL_FLASH = "gemini-2-5-flash"
FIXED_TEMP  = "0.0"


# ─────────────────────────────────────────────────────────────────────────────
# Robust canonicalization (as you provided)
# ─────────────────────────────────────────────────────────────────────────────

_CURLY_TO_STRAIGHT = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'",
    "“": '"', "”": '"', "„": '"', "‟": '"',
})

# def _canon_text(s: str) -> str:
#     s = unicodedata.normalize("NFKC", s)
#     s = s.translate(_CURLY_TO_STRAIGHT)
#     s = s.strip()
#     s = re.sub(r"\s+", " ", s)
#     return s

def _canon_text(s: str) -> str:
    if not isinstance(s, str): return ""
    s = unicodedata.normalize("NFKC", s)
    s = s.translate(_CURLY_TO_STRAIGHT)
    s = s.strip().upper() 
    s = re.sub(r"\s+", " ", s)
    return s

# def _to_label_set(x) -> frozenset[str]:
#     if x is None or (isinstance(x, float) and pd.isna(x)) or pd.isna(x):
#         return frozenset()

#     if isinstance(x, (list, tuple, set)):
#         items = [str(i) for i in x if str(i).strip() != ""]
#         return frozenset(_canon_text(i) for i in items if _canon_text(i) != "")

#     s = _canon_text(str(x))
#     if s == "":
#         return frozenset()

#     if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")) or (s.startswith("{") and s.endswith("}")):
#         try:
#             parsed = ast.literal_eval(s)
#             if isinstance(parsed, (list, tuple, set)):
#                 items = [str(i) for i in parsed if str(i).strip() != ""]
#                 return frozenset(_canon_text(i) for i in items if _canon_text(i) != "")
#             if isinstance(parsed, str):
#                 parsed_s = _canon_text(parsed)
#                 return frozenset([parsed_s]) if parsed_s else frozenset()
#         except Exception:
#             pass

#     if "," in s:
#         parts = [_canon_text(p) for p in s.split(",")]
#         parts = [p for p in parts if p]
#         return frozenset(parts)

#     return frozenset([s])

def _to_label_set(x) -> frozenset[str]:
    if x is None or (isinstance(x, float) and pd.isna(x)) or pd.isna(x):
        return frozenset()

    if isinstance(x, (list, tuple, set)):
        items = [str(i) for i in x if str(i).strip() != ""]
        return frozenset(_canon_text(i) for i in items if _canon_text(i) != "")

    s = _canon_text(str(x))
    if not s:
        return frozenset()

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")) or (s.startswith("{") and s.endswith("}")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set)):
                items = [str(i) for i in parsed if str(i).strip() != ""]
                return frozenset(_canon_text(i) for i in items if _canon_text(i) != "")
            if isinstance(parsed, str):
                parsed_s = _canon_text(parsed)
                return frozenset([parsed_s]) if parsed_s else frozenset()
        except Exception:
            pass # literal_eval failed (e.g., missing quotes inside brackets)

    # FIX 2: If literal_eval failed, strip the literal brackets BEFORE splitting
    # so we don't end up with elements like "[REPORTED"
    s_clean = re.sub(r"^\[|\]$", "", s).strip()

    if "," in s_clean:
        parts = [_canon_text(p) for p in s_clean.split(",")]
        parts = [p for p in parts if p]
        return frozenset(parts)

    return frozenset([_canon_text(s_clean)])

def _parse_path(p: Path) -> dict:
    parts = p.parts
    i = parts.index("output")
    task, model = parts[i + 1], parts[i + 2]
    temperature = "unknown"
    for part in parts[i + 3:]:
        m = re.match(r"temperature_([0-9]+(?:\.[0-9]+)?)", part)
        if m:
            temperature = m.group(1)
            break
    return {"task": task, "model": model, "temperature": temperature}

# def load_merged() -> pd.DataFrame:
#     gt = pd.read_csv(GT_PATH, sep=";", dtype={"paper_id": str})

#     gt_long = (
#         gt[["paper_id"] + list(TASK_TO_GT_COL.values())]
#         .melt(id_vars="paper_id", var_name="gt_col", value_name="gt_raw")
#     )
#     inv = {v: k for k, v in TASK_TO_GT_COL.items()}
#     gt_long["task"] = gt_long["gt_col"].map(inv)
#     gt_long = gt_long.drop(columns="gt_col")
#     gt_long["gt_set"] = gt_long["gt_raw"].map(_to_label_set)
#     gt_long = gt_long.drop(columns="gt_raw")
#     gt_long = gt_long[gt_long["gt_set"].map(len) > 0].copy()

#     # Conservative union if duplicates ever happen
#     gt_long = (
#         gt_long.groupby(["paper_id", "task"], as_index=False)
#         .agg(gt_set=("gt_set", lambda sets: frozenset().union(*sets)))
#     )

#     frames = []
#     for f in sorted(OUTPUT_DIR.rglob("final_*.tsv")):
#         try:
#             df = pd.read_csv(f, sep="\t", dtype={"paper_id": str})
#             meta = _parse_path(f)
#             df["task"]        = meta["task"]
#             df["model"]       = meta["model"]
#             df["temperature"] = meta["temperature"]
#             df["pred_set"]    = df["classification"].map(_to_label_set)
#             df["source_file"] = str(f)
#             frames.append(df[["paper_id", "task", "model", "temperature", "pred_set", "source_file"]])
#         except Exception as e:
#             print(f"[WARN] Skipped {f}: {e}")

#     pred = pd.concat(frames, ignore_index=True)
#     pred = pred[pred["task"].isin(TASK_TO_GT_COL)].copy()

#     merged = pred.merge(gt_long, on=["paper_id", "task"], how="inner")
#     merged["is_incorrect"] = merged["pred_set"] != merged["gt_set"]
#     return merged

def load_merged() -> pd.DataFrame:
    gt = pd.read_csv(GT_PATH, sep="\t", dtype={"paper_id": str})
    
    # FIX 3: Strip whitespace from GT paper IDs to ensure clean merges
    gt["paper_id"] = gt["paper_id"].str.strip()

    gt_long = (
        gt[["paper_id"] + list(TASK_TO_GT_COL.values())]
        .melt(id_vars="paper_id", var_name="gt_col", value_name="gt_raw")
    )
    inv = {v: k for k, v in TASK_TO_GT_COL.items()}
    gt_long["task"] = gt_long["gt_col"].map(inv)
    gt_long = gt_long.drop(columns="gt_col")
    gt_long["gt_set"] = gt_long["gt_raw"].map(_to_label_set)
    gt_long = gt_long.drop(columns="gt_raw")
    
    # Note: If GT is empty/NaN, it is dropped here. If a paper is missing from 
    # "always wrong", verify it actually has a valid label in the GT!
    gt_long = gt_long[gt_long["gt_set"].map(len) > 0].copy()

    # show what has been dropped due to empty GT sets (sometimes useful for debugging)
    dropped_gt = gt_long[gt_long["gt_set"].map(len) == 0]
    if not dropped_gt.empty:
        print(f"[INFO] Dropping {len(dropped_gt)} rows with empty GT sets:")
        display(dropped_gt.head(10))

    gt_long = (
        gt_long.groupby(["paper_id", "task"], as_index=False)
        .agg(gt_set=("gt_set", lambda sets: frozenset().union(*sets)))
    )

    frames = []
    for f in sorted(OUTPUT_DIR.rglob("final_*.tsv")):
        # Tip: Ensure rglob("final_*.tsv") isn't accidentally loading old backup files 
        # (e.g., "final_v2.tsv"). This would inflate n_runs and ruin the 'always wrong' logic.
        try:
            df = pd.read_csv(f, sep="\t", dtype={"paper_id": str})
            # FIX 3: Strip whitespace from Pred paper IDs
            df["paper_id"] = df["paper_id"].str.strip()
            
            meta = _parse_path(f)
            df["task"]        = meta["task"]
            df["model"]       = meta["model"]
            df["temperature"] = meta["temperature"]
            df["pred_set"]    = df["classification"].map(_to_label_set)
            df["source_file"] = str(f)
            frames.append(df[["paper_id", "task", "model", "temperature", "pred_set", "source_file"]])
        except Exception as e:
            print(f"[WARN] Skipped {f}: {e}")

    pred = pd.concat(frames, ignore_index=True)
    pred = pred[pred["task"].isin(TASK_TO_GT_COL)].copy()

    merged = pred.merge(gt_long, on=["paper_id", "task"], how="inner")
    merged["is_incorrect"] = merged["pred_set"] != merged["gt_set"]
    return merged


# ─────────────────────────────────────────────────────────────────────────────
# A. Within-config consistency across runs (frozenset-safe)
# ─────────────────────────────────────────────────────────────────────────────

def within_config_consistency(merged: pd.DataFrame) -> pd.DataFrame:
    """
    For each (task, model, temperature), for each paper:
      stable if identical prediction-set across runs (n_unique_preds == 1)

    Summary returned per (task, model, temperature):
      n_papers
      n_stable / n_unstable / pct_stable
      n_stable_correct / n_stable_wrong
    """
    df = merged.copy()
    df["pred_key"] = df["pred_set"].map(lambda s: "|".join(sorted(s)))

    per_paper = (
        df.groupby(["task", "model", "temperature", "paper_id"], as_index=False)
          .agg(
              n_runs=("pred_key", "count"),
              n_unique_preds=("pred_key", "nunique"),
              gt_set=("gt_set", "first"),
              # "always_wrong" means wrong in every run
              always_wrong=("is_incorrect", "min"),
              # "ever_wrong" means wrong at least once (sometimes useful to inspect)
              ever_wrong=("is_incorrect", "max"),
          )
    )
    per_paper["is_stable"] = per_paper["n_unique_preds"] == 1

    summary = (
        per_paper.groupby(["task", "model", "temperature"], as_index=False)
                 .agg(
                     n_papers=("paper_id", "count"),
                     n_stable=("is_stable", "sum"),
                     n_unstable=("is_stable", lambda s: (~s).sum()),
                     n_stable_wrong=("always_wrong", lambda s: ((per_paper.loc[s.index, "is_stable"]) & (s)).sum()),
                     n_stable_correct=("always_wrong", lambda s: ((per_paper.loc[s.index, "is_stable"]) & (~s)).sum()),
                 )
    )
    summary["pct_stable"] = (100 * summary["n_stable"] / summary["n_papers"]).round(1)
    return summary.sort_values(["task", "model", "temperature"]).reset_index(drop=True)


def unstable_papers(merged: pd.DataFrame, task: str, model: str, temperature: str) -> pd.DataFrame:
    """
    List papers where the prediction-set changes across runs.
    Shows GT set + the set of unique prediction-sets observed.
    """
    df = merged[
        (merged["task"] == task) &
        (merged["model"] == model) &
        (merged["temperature"] == temperature)
    ].copy()

    df["pred_key"] = df["pred_set"].map(lambda s: "|".join(sorted(s)))
    df["gt_key"] = df["gt_set"].map(lambda s: "|".join(sorted(s)))

    per = (
        df.groupby(["paper_id"], as_index=False)
          .agg(
              gt_key=("gt_key", "first"),
              n_runs=("pred_key", "count"),
              n_unique_preds=("pred_key", "nunique"),
              preds=("pred_key", lambda s: ", ".join(sorted(s.unique()))),
          )
    )
    out = per[per["n_unique_preds"] > 1].copy()
    out["gt_label"] = out["gt_key"].map(lambda k: k.split("|") if k else [])
    return out.drop(columns=["gt_key"]).sort_values("paper_id").reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# B. Containment: pro vs flash at a fixed temperature (ever-wrong sets)
# ─────────────────────────────────────────────────────────────────────────────

def containment_pro_vs_flash(merged: pd.DataFrame, temperature: str = FIXED_TEMP) -> pd.DataFrame:
    """
    At fixed temperature, for each task:
      pro_error_set   = papers where pro is wrong in at least one run
      flash_error_set = papers where flash is wrong in at least one run
      compute containment and Jaccard
    """
    subset = merged[merged["temperature"] == temperature].copy()

    rows = []
    for task in TASK_TO_GT_COL:
        task_df = subset[subset["task"] == task]

        def ever_wrong_set(model: str) -> set[str]:
            m = task_df[task_df["model"] == model]
            if m.empty:
                return set()
            ever_wrong = m.groupby("paper_id")["is_incorrect"].max()
            return set(ever_wrong[ever_wrong].index)

        pro = ever_wrong_set(MODEL_PRO)
        flash = ever_wrong_set(MODEL_FLASH)

        shared = pro & flash
        union = pro | flash

        rows.append({
            "task": task,
            "temperature": temperature,
            "n_pro_errors": len(pro),
            "n_flash_errors": len(flash),
            "n_shared": len(shared),
            "pro_in_flash_%": round(100 * len(shared) / len(pro), 1) if pro else None,
            "flash_in_pro_%": round(100 * len(shared) / len(flash), 1) if flash else None,
            "jaccard_%": round(100 * len(shared) / len(union), 1) if union else None,
            "n_pro_only": len(pro - flash),
            "n_flash_only": len(flash - pro),
        })

    return pd.DataFrame(rows)


# ─────────────────────────────────────────────────────────────────────────────
# C. Pro @ fixed temperature: papers always wrong (your version, unchanged)
# ─────────────────────────────────────────────────────────────────────────────

def pro_always_wrong(merged: pd.DataFrame, temperature: str = FIXED_TEMP) -> pd.DataFrame:
    subset = merged[
        (merged["model"] == MODEL_PRO) &
        (merged["temperature"] == temperature)
    ].copy()

    per_paper = (
        subset.groupby(["task", "paper_id"], as_index=False)
        .agg(
            gt_set=("gt_set", "first"),
            n_runs=("is_incorrect", "count"),
            wrong_runs=("is_incorrect", "sum"),
        )
    )
    always_wrong = per_paper[per_paper["wrong_runs"] == per_paper["n_runs"]].copy()
    if always_wrong.empty:
        return always_wrong.assign(top_wrong_pred=pd.Series(dtype=object))

    wrong_only = subset[subset["is_incorrect"]].copy()
    wrong_only["pred_key"] = wrong_only["pred_set"].map(lambda s: "|".join(sorted(s)))

    top_wrong = (
        wrong_only.groupby(["task", "paper_id"])["pred_key"]
        .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else "")
        .rename("top_wrong_key")
        .reset_index()
    )
    top_wrong["top_wrong_pred"] = top_wrong["top_wrong_key"].map(
        lambda k: frozenset(k.split("|")) if k else frozenset()
    )
    top_wrong = top_wrong.drop(columns=["top_wrong_key"])

    result = always_wrong.merge(top_wrong, on=["task", "paper_id"], how="left")

    contradictions = result[result["top_wrong_pred"] == result["gt_set"]]
    if not contradictions.empty:
        print("[WARN] Contradictions remain (top_wrong_pred == gt_set). Inspect:")
        display(contradictions.head(20))

    return result.sort_values(["task", "paper_id"]).reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# Run
# ─────────────────────────────────────────────────────────────────────────────

merged = load_merged()

# A
print("\n=== A. Within-config consistency across runs ===\n")
consistency = within_config_consistency(merged)
display(consistency)
consistency.to_csv("within_config_consistency.csv", index=False)

# Optional drill-down:
# display(unstable_papers(merged, task="geo", model=MODEL_PRO, temperature=FIXED_TEMP))

# B
print(f"\n=== B. Containment: pro vs flash (temperature = {FIXED_TEMP}) ===\n")
containment = containment_pro_vs_flash(merged, temperature=FIXED_TEMP)
display(containment)
containment.to_csv("containment_pro_vs_flash.csv", index=False)

# C
print(f"\n=== C. Papers pro (T={FIXED_TEMP}) gets wrong in every run ===\n")
pro_errors = pro_always_wrong(merged, temperature=FIXED_TEMP)
pro_errors.to_csv("pro_always_wrong.csv", index=False)

for task, grp in pro_errors.groupby("task"):
    print(f"\n-- {task} ({len(grp)} papers always wrong) --")
    display(
        grp.drop(columns="task")
           .assign(
               gt_label=lambda d: d["gt_set"].map(lambda s: list(sorted(s))),
               top_pred=lambda d: d["top_wrong_pred"].map(lambda s: list(sorted(s))),
           )
           .drop(columns=["gt_set", "top_wrong_pred"])
           .reset_index(drop=True)
    )

ValueError: No objects to concatenate

In [ ]:
# """
# Model Classification Error Analysis
# ====================================
# Compares classification errors made by different models across temperature
# settings and four tasks (geo, data-type, data-accessibility, paper-type).

# Story structure
# ---------------
# 1. Load & merge          – ground truth + all prediction files
# 2. Overview              – total errors per task × model × temperature
# 3. Hard papers           – which papers are most often misclassified?
# 4. Per-task heatmaps     – paper × config error grids
# 5. Config correlations   – do different configs fail on the same papers?
#                            (with statistical significance via Fisher's exact test)
# """

# from __future__ import annotations

# import re
# from pathlib import Path
# from typing import Dict, List, Optional, Tuple

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from matplotlib.colors import BoundaryNorm, ListedColormap
# from scipy import stats


# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──
# # Configuration
# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──

# ROOT       = Path(".")
# GT_PATH    = ROOT / "sampled_papers_full.csv"
# OUTPUT_DIR = ROOT / "output"

# TASK_TO_GT_COL = {
#     "geo":                "geo_classification",
#     "data-type":          "data_type_classification",
#     "data-accessibility": "availability_classification",
#     "paper-type":         "ptype_classification",
# }

# # How to aggregate multiple runs (final_n.tsv files) for one paper + config:
# #   "count" – number of runs in which the paper was wrong  (0 … R)
# #   "any"   – 1 if wrong in at least one run, else 0 (binary)
# AGG_MODE = "count"


# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──
# # 1. Data loading helpers
# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──

# def normalize_label(x) -> str:
#     """Strip whitespace/NaN; collapse interior spaces."""
#     if pd.isna(x):
#         return ""
#     return re.sub(r"\s+", " ", str(x).strip())


# def _parse_path_metadata(p: Path) -> Dict[str, str]:
#     """
#     Extract task / model / temperature from a results path of the form:
#         output/{task}/{model}/temperature_{t}/.../final_n.tsv
#     """
#     parts = p.parts
#     try:
#         i = parts.index("output")
#     except ValueError:
#         raise ValueError(f"Path lacks an 'output' segment: {p}")
#     if len(parts) <= i + 3:
#         raise ValueError(f"Path too shallow to parse metadata: {p}")

#     task  = parts[i + 1]
#     model = parts[i + 2]

#     # Find temperature_X anywhere from i+3 onward (robust to extra nesting)
#     temperature = "unknown"
#     for part in parts[i + 3:]:
#         m = re.match(r"temperature_(?P<t>[0-9]+(?:\.[0-9]+)?)", part)
#         if m:
#             temperature = m.group("t")
#             break

#     return {"task": task, "model": model, "temperature": temperature}


# def _load_single_tsv(path: Path) -> pd.DataFrame:
#     """Read one final_*.tsv and tag it with task/model/temperature metadata."""
#     df = pd.read_csv(path, sep="\t", dtype={"paper_id": str})
#     required = {"paper_id", "classification"}
#     if not required.issubset(df.columns):
#         raise ValueError(f"{path}: missing columns {required - set(df.columns)}")

#     meta = _parse_path_metadata(path)
#     df = df.assign(
#         task        = meta["task"],
#         model       = meta["model"],
#         temperature = meta["temperature"],
#         pred_label  = df["classification"].map(normalize_label),
#     )
#     keep = ["paper_id", "task", "model", "temperature", "pred_label"]
#     for opt in ("confidence", "evidence"):
#         if opt in df.columns:
#             keep.append(opt)
#     return df[keep]


# def load_predictions(output_dir: Path = OUTPUT_DIR) -> pd.DataFrame:
#     """Discover and load all final_*.tsv files under *output_dir*."""
#     files = sorted(output_dir.rglob("final_*.tsv"))
#     if not files:
#         raise FileNotFoundError(f"No final_*.tsv found under {output_dir.resolve()}")

#     frames, skipped = [], []
#     for f in files:
#         try:
#             frames.append(_load_single_tsv(f))
#         except Exception as exc:
#             skipped.append((f, exc))

#     for f, exc in skipped:
#         print(f"[WARN] Skipped {f}: {exc}")
#     if not frames:
#         raise RuntimeError("All TSV files failed to load.")
#     return pd.concat(frames, ignore_index=True)


# def load_ground_truth(gt_path: Path = GT_PATH) -> pd.DataFrame:
#     """Load and normalise the ground-truth CSV."""
#     gt = pd.read_csv(gt_path, sep="\t", dtype={"paper_id": str})
#     missing = [c for c in TASK_TO_GT_COL.values() if c not in gt.columns]
#     if missing:
#         raise KeyError(f"Ground-truth file is missing columns: {missing}")
#     for col in TASK_TO_GT_COL.values():
#         gt[col] = gt[col].map(normalize_label)
#     return gt


# def build_merged(
#     gt_path: Path = GT_PATH,
#     output_dir: Path = OUTPUT_DIR,
# ) -> pd.DataFrame:
#     """
#     Join predictions with ground truth (long form).

#     Returns one row per (paper_id, task, model, temperature, run) with:
#       gt_label    – ground-truth label
#       pred_label  – model prediction
#       is_incorrect – True when pred ≠ gt (and gt is not missing)
#     """
#     gt   = load_ground_truth(gt_path)
#     pred = load_predictions(output_dir)
#     pred = pred[pred["task"].isin(TASK_TO_GT_COL)].copy()

#     # Pivot GT to long form: one row per (paper_id, task)
#     inv     = {v: k for k, v in TASK_TO_GT_COL.items()}
#     gt_long = (
#         gt[["paper_id"] + list(TASK_TO_GT_COL.values())]
#         .melt(id_vars="paper_id", var_name="gt_col", value_name="gt_label")
#         .assign(task=lambda d: d["gt_col"].map(inv))
#         .drop(columns="gt_col")
#     )
#     gt_long["gt_label"] = gt_long["gt_label"].map(normalize_label)

#     merged = pred.merge(gt_long, on=["paper_id", "task"], how="left")
#     merged["gt_missing"]  = merged["gt_label"].isna()
#     merged["is_incorrect"] = (~merged["gt_missing"]) & (merged["pred_label"] != merged["gt_label"])
#     return merged


# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──
# # 2. Overview: aggregate error counts
# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──

# def error_summary(merged: pd.DataFrame) -> pd.DataFrame:
#     """
#     Per (task, model, temperature): total predictions, errors, and error rate.
#     Useful as a first look at which configs struggle most.
#     """
#     return (
#         merged.groupby(["task", "model", "temperature"], as_index=False)
#         .agg(
#             n_predictions = ("is_incorrect", "size"),
#             n_incorrect   = ("is_incorrect", "sum"),
#             n_missing_gt  = ("gt_missing",   "sum"),
#         )
#         .assign(error_rate=lambda d: d["n_incorrect"] / d["n_predictions"])
#         .sort_values(["task", "error_rate"], ascending=[True, False])
#     )


# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──
# # 3. Hard papers
# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──

# def hard_papers(
#     merged: pd.DataFrame,
#     task: Optional[str] = None,
#     agg_mode: str = AGG_MODE,
#     top_k: int = 30,
# ) -> pd.DataFrame:
#     """
#     Rank papers by how often they are misclassified across all configs.

#     Parameters
#     ----------
#     task     : restrict to one task; None = aggregate across all tasks.
#     agg_mode : "any" counts a paper once per config (binary),
#                "count" sums up the number of wrong runs per config.
#     top_k    : number of hardest papers to return.
#     """
#     df = merged if task is None else merged[merged["task"] == task]

#     if agg_mode == "any":
#         agg_fn = "max"   # 0 or 1 per (paper, config)
#     elif agg_mode == "count":
#         agg_fn = "sum"   # number of wrong runs per (paper, config)
#     else:
#         raise ValueError("agg_mode must be 'any' or 'count'")

#     per_config = (
#         df.groupby(["paper_id", "task", "model", "temperature"], as_index=False)
#         .agg(err=("is_incorrect", agg_fn))
#     )
#     totals = (
#         per_config.groupby("paper_id", as_index=False)["err"]
#         .sum()
#         .rename(columns={"err": "total_errors"})
#         .sort_values("total_errors", ascending=False)
#         .head(top_k)
#     )
#     return totals


# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──
# # 4. Per-task paper × config error matrix + heatmap
# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──

# def paper_config_matrix(
#     merged: pd.DataFrame,
#     task: str,
#     agg_mode: str = AGG_MODE,
# ) -> pd.DataFrame:
#     """
#     Matrix with paper_id as rows and (model, temperature) as columns.
#     Values are error counts (or 0/1 if agg_mode="any").
#     """
#     df  = merged[merged["task"] == task]
#     agg = "sum" if agg_mode == "count" else "max"

#     mat = (
#         df.groupby(["paper_id", "model", "temperature"], as_index=False)
#         .agg(err=("is_incorrect", agg))
#         .pivot_table(index="paper_id", columns=["model", "temperature"],
#                      values="err", fill_value=0, aggfunc="sum")
#         .astype(int)
#     )
#     return mat


# def plot_task_heatmap(
#     merged: pd.DataFrame,
#     task: str,
#     top_k: int = 80,
#     agg_mode: str = AGG_MODE,
#     figsize: Tuple[int, int] = (12, 18),
#     annotate: bool = True,
# ) -> Tuple[pd.DataFrame, pd.Series]:
#     """
#     Discrete heatmap: rows = papers (sorted by total errors), cols = configs.
#     Column headers include the per-config error total.

#     Returns the matrix and per-column totals for downstream use.
#     """
#     mat       = paper_config_matrix(merged, task=task, agg_mode=agg_mode)
#     row_order = mat.sum(axis=1).sort_values(ascending=False)
#     mat       = mat.loc[row_order.index].head(top_k)
#     col_totals = mat.sum(axis=0)

#     arr  = mat.to_numpy(dtype=float)
#     vmax = int(arr.max()) if arr.size else 0

#     boundaries = np.arange(-0.5, vmax + 1.5, 1.0)
#     base_cmap  = plt.get_cmap("viridis")
#     cmap       = ListedColormap(base_cmap(np.linspace(0, 1, max(vmax + 1, 1))))
#     norm       = BoundaryNorm(boundaries, cmap.N)

#     # fig, ax = plt.subplots(figsize=figsize)
#     # im = ax.imshow(arr, aspect="auto", cmap=cmap, norm=norm)

#     # ax.set_yticks(np.arange(mat.shape[0]))
#     # ax.set_yticklabels(mat.index.tolist(), fontsize=8)

#     # col_labels = [f"{m}\nT={t}\nerr={int(col_totals[(m, t)])}"
#     #               for m, t in mat.columns.tolist()]
#     # ax.set_xticks(np.arange(mat.shape[1]))
#     # ax.set_xticklabels(col_labels, fontsize=9)

#     # ax.set_xlabel("Model / Temperature  (err = total column errors)")
#     # ax.set_ylabel(f"paper_id  (top {top_k} by error count)")
#     # mode_label = "wrong-run count" if agg_mode == "count" else "ever-wrong (binary)"
#     # ax.set_title(f"Classification errors — task: {task}  [{mode_label}]")

#     # if annotate:
#     #     for i in range(arr.shape[0]):
#     #         for j in range(arr.shape[1]):
#     #             v = int(arr[i, j])
#     #             if v:
#     #                 colour = "white" if vmax and v >= 0.6 * vmax else "black"
#     #                 ax.text(j, i, str(v), ha="center", va="center",
#     #                         fontsize=7, color=colour)

#     # cbar = fig.colorbar(im, ax=ax, shrink=0.8)
#     # cbar.set_label("# errors")
#     # cbar.set_ticks(np.arange(0, vmax + 1))
#     # plt.tight_layout()
#     # plt.show()

#     return mat, col_totals


# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──
# # 5. Config correlations with significance testing
# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──

# def config_correlation_table(
#     mat: pd.DataFrame,
#     alpha: float = 0.05,
# ) -> pd.DataFrame:
#     """
#     For every pair of configs (columns of *mat*), compute:
#       - phi coefficient  (Pearson on binary error vectors)
#       - two-tailed p-value via Fisher's exact test on the 2×2 contingency table
#       - significance flag at *alpha*
#       - Jaccard similarity of error sets
#       - containment of A's errors inside B and vice-versa

#     *mat* is the paper × config matrix (values treated as binary: >0 → error).

#     Returns a long-form DataFrame, one row per config pair.
#     """
#     bin_mat = (mat > 0).astype(int)
#     cols    = list(bin_mat.columns)
#     rows    = []

#     for i, col_a in enumerate(cols):
#         for col_b in cols[i + 1:]:
#             a = bin_mat[col_a]
#             b = bin_mat[col_b]

#             # 2×2 contingency table: (a=1 & b=1), (a=1 & b=0), etc.
#             n11 = int(((a == 1) & (b == 1)).sum())
#             n10 = int(((a == 1) & (b == 0)).sum())
#             n01 = int(((a == 0) & (b == 1)).sum())
#             n00 = int(((a == 0) & (b == 0)).sum())
#             contingency = [[n11, n10], [n01, n00]]

#             _, p_value = stats.fisher_exact(contingency, alternative="two-sided")

#             # Phi / Pearson (nan-safe)
#             phi = float(np.corrcoef(a, b)[0, 1]) if a.nunique() > 1 and b.nunique() > 1 else np.nan

#             # Set-based metrics
#             A     = set(mat.index[a == 1])
#             B     = set(mat.index[b == 1])
#             inter = A & B
#             union = A | B
#             jaccard         = len(inter) / len(union) if union else np.nan
#             contain_a_in_b  = len(inter) / len(A)     if A      else np.nan
#             contain_b_in_a  = len(inter) / len(B)     if B      else np.nan

#             rows.append({
#                 "config_A":         col_a,
#                 "config_B":         col_b,
#                 "phi":              round(phi, 4),
#                 "p_value":          round(p_value, 6),
#                 "significant":      p_value < alpha,
#                 "jaccard":          round(jaccard, 4)        if not np.isnan(jaccard) else np.nan,
#                 "contain_A_in_B":   round(contain_a_in_b, 4) if not np.isnan(contain_a_in_b) else np.nan,
#                 "contain_B_in_A":   round(contain_b_in_a, 4) if not np.isnan(contain_b_in_a) else np.nan,
#                 "n_errors_A":       len(A),
#                 "n_errors_B":       len(B),
#                 "n_shared_errors":  len(inter),
#             })

#     return pd.DataFrame(rows).sort_values("phi", ascending=False)


# def plot_phi_heatmap(
#     mat: pd.DataFrame,
#     task: str,
#     figsize: Tuple[int, int] = (8, 6),
# ) -> None:
#     """
#     Symmetric heatmap of pairwise phi correlations between configs.
#     Cells are annotated with phi; p<0.05 cells get a border marker (*).
#     """
#     bin_mat  = (mat > 0).astype(int)
#     corr_df  = bin_mat.corr(method="pearson")
#     corr_tbl = config_correlation_table(mat)

#     # Build significance mask matrix
#     n    = len(corr_df)
#     cols = corr_df.columns.tolist()
#     sig  = np.zeros((n, n), dtype=bool)
#     for _, row in corr_tbl.iterrows():
#         if row["significant"]:
#             try:
#                 i = cols.index(row["config_A"])
#                 j = cols.index(row["config_B"])
#                 sig[i, j] = sig[j, i] = True
#             except ValueError:
#                 pass

#     arr = corr_df.to_numpy(dtype=float)
#     fig, ax = plt.subplots(figsize=figsize)
#     im = ax.imshow(arr, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")

#     labels = [f"{m}\nT={t}" for m, t in cols]
#     ax.set_xticks(range(n)); ax.set_xticklabels(labels, fontsize=8)
#     ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=8)
#     ax.set_title(f"Phi correlations between configs — task: {task}\n(* = p < 0.05)")

#     for i in range(n):
#         for j in range(n):
#             txt = f"{arr[i,j]:.2f}" + ("*" if sig[i, j] else "")
#             ax.text(j, i, txt, ha="center", va="center", fontsize=7,
#                     color="white" if abs(arr[i, j]) > 0.5 else "black")

#     fig.colorbar(im, ax=ax, shrink=0.7, label="phi")
#     plt.tight_layout()
#     plt.show()


# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──
# # Run the full analysis
# # ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ── ──

# if __name__ == "__main__":

#     # ── Step 1: Load data ──────────────────────────────────────────────────
#     print("Loading data…")
#     merged = build_merged()
#     print(f"  {len(merged):,} prediction rows across "
#           f"{merged['paper_id'].nunique()} papers, "
#           f"{merged['task'].nunique()} tasks, "
#           f"{merged['model'].nunique()} models.\n")

#     # # ── Step 2: Overview table ─────────────────────────────────────────────
#     # print("=== Error rate overview (task × model × temperature) ===")
#     # summary = error_summary(merged)
#     # display(summary)
#     # summary.to_csv("error_summary.csv", index=False)

#     # # ── Step 3: Hard papers (all tasks combined) ───────────────────────────
#     # print("\n=== Hardest papers across all tasks ===")
#     # hard = hard_papers(merged, task=None, agg_mode="any", top_k=30)
#     # display(hard)
#     # hard.to_csv("hard_papers.csv", index=False)

#     # ── Steps 4 & 5: Per-task deep dive ────────────────────────────────────
#     for task in TASK_TO_GT_COL:
#         print(f"\n{'='*60}")
#         print(f"  Task: {task}")
#         print(f"{'='*60}")

#         # # 4a. Heatmap: which papers fail most, and under which configs?
#         mat, col_totals = plot_task_heatmap(
#             merged, task=task, top_k=80, agg_mode=AGG_MODE,
#             figsize=(12, 18), annotate=True,
#         )

#         # 4b. Hard papers for this task specifically
#         print(f"\nTop-10 hardest papers for task '{task}':")
#         display(hard_papers(merged, task=task, agg_mode="any", top_k=10))

#         # 5a. Pairwise config correlation (phi + p-values)
#         if mat.shape[1] >= 2:
#             print(f"\nPairwise config correlations (Fisher p-value, task='{task}'):")
#             corr_tbl = config_correlation_table(mat)
#             display(corr_tbl)
#             corr_tbl.to_csv(f"config_correlations_{task}.csv", index=False)

#             # 5b. Phi heatmap
#             plot_phi_heatmap(mat, task=task)
#         else:
#             print(f"  Fewer than 2 configs found for task '{task}' — skipping correlation.")
